# FarmTech Solutions — Fase 6: Visão Computacional
## Entrega 2 — Comparação de abordagens: YOLO customizada × YOLO tradicional × CNN do zero

**Grupo:** Grupo 3
**Integrantes:** Gerson Ferreira da Graça (RM 569624), Lucas Braga (RM 568712), Carlos Leonardo Mazieri (RM 572809), Ryann Pinto (RM 571003)
**Objetos:** Tomate (classe 0) e Pimentão (classe 1)

---

### Objetivo desta entrega

Sobre a **mesma base de 80 imagens da Entrega 1** (32 treino / 4 validação / 4 teste por classe), comparar três abordagens de visão computacional:

| | Abordagem | O que é neste notebook | Treino no nosso dataset? |
|:--|:--|:--|:--:|
| **A** | YOLO customizada | O `best.pt` da Entrega 1: YOLOv8n pré-treinado no COCO e **ajustado** (fine-tuning, 60 épocas) com as nossas caixas de tomate e pimentão. Reaproveitado, não é treinado de novo. | Sim (Entrega 1) |
| **B** | YOLO tradicional | O YOLOv8n **padrão**, com os pesos genéricos do COCO (80 classes do dia a dia), usado como vem, **sem nenhum treino** no nosso dataset. | Não |
| **C** | CNN do zero | Rede convolucional com arquitetura própria, pesos aleatórios no início, treinada só com as nossas imagens para **classificar a imagem inteira** (sem caixas). | Sim |

As três são avaliadas nas **mesmas 8 imagens de teste** e comparadas nos quatro critérios pedidos: **facilidade de uso/integração**, **precisão**, **tempo de treinamento/customização** e **tempo de inferência**.

> **Por que "YOLO tradicional" = modelo pré-treinado sem ajuste:** a diferença entre a Entrega 1 e a "YOLO tradicional" é a customização. A Entrega 1 já parte do YOLOv8n pré-treinado e o adapta ao nosso problema; a versão tradicional é o mesmo detector usado como sai da caixa, reconhecendo apenas as classes do COCO. Treinar outro YOLO com outra configuração produziria praticamente o mesmo modelo da Entrega 1 (ela já usa os hiperparâmetros padrão) e não acrescentaria nada à comparação.

## 1. Setup do ambiente

Conectando ao Google Drive e instalando as dependências. Use um ambiente com **GPU T4** (Ambiente de execução > Alterar tipo de ambiente de execução) — os tempos de treino e de inferência deste notebook são medidos nesse hardware.

In [ ]:
# Conectar ao Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho base do projeto no Drive (o mesmo da Entrega 1).
# Quem NÃO é o dono da pasta precisa, uma única vez, criar o atalho:
# "Compartilhados comigo" → botão direito em FarmTech_Fase6 → "Organizar" → "Adicionar atalho" → "Meu Drive".
BASE_PATH = '/content/drive/MyDrive/FarmTech_Fase6'

# Identificação de quem está rodando o notebook.
# Cada integrante troca para o próprio primeiro nome (minúsculo, sem acento) ANTES de rodar.
# Tudo o que esta entrega grava vai para Resultados/entrega2_<EXECUTOR>/.
EXECUTOR = 'gerson'

# Modelo da Entrega 1 (abordagem A): execução oficial registrada no notebook da Entrega 1
# (60 épocas, pasta treino-2). É fixo de propósito — todos comparam contra o MESMO modelo,
# independentemente de quem roda este notebook.
PASTA_TREINO_E1 = f'{BASE_PATH}/Resultados/epocas_60_gerson/treino-2'
PESOS_E1 = f'{PASTA_TREINO_E1}/weights/best.pt'

# Pasta de saída desta entrega
PASTA_E2 = f'{BASE_PATH}/Resultados/entrega2_{EXECUTOR}'

In [ ]:
# Checar se o ambiente de execução tem GPU
import torch
print("GPU disponível:", torch.cuda.is_available())
print("Dispositivo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (mais lento — considere trocar o ambiente de execução para GPU T4 em Ambiente de execução > Alterar tipo de ambiente de execução)")

In [ ]:
# Instalar o YOLO (Ultralytics) na mesma versão usada na execução oficial da Entrega 1.
# O TensorFlow (usado na CNN) já vem instalado no Colab.
!pip install ultralytics==8.4.170 -q

import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from sklearn.metrics import classification_report, confusion_matrix
from ultralytics import YOLO

# O TensorFlow, por padrão, reserva toda a memória da GPU ao ser usado pela primeira vez — o que
# deixaria o YOLO (PyTorch) sem memória. Com "memory growth" ele reserva só o que precisa.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

os.makedirs(PASTA_E2, exist_ok=True)

import ultralytics
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| tensorflow', tf.__version__)
print('Saídas desta entrega em:', PASTA_E2)

## 2. Dataset — as mesmas 80 imagens da Entrega 1

Nenhuma imagem é copiada ou movida nesta entrega. As duas organizações já existem no Drive:

- **Por classe** (como a equipe montou): `Dataset_dividido/{Tomate,Pimentao}/{Treino,Validacao,Teste}/` — é daqui que a **CNN** lê, porque para classificação a "etiqueta" de cada imagem é simplesmente a pasta da classe.
- **Formato YOLO** (gerado na seção 2.1 do notebook da Entrega 1): `Dataset_dividido/{treino,validacao,teste}/{images,labels}/` — usado pelas duas **YOLO**, porque a avaliação de detecção precisa das caixas rotuladas.

A célula abaixo confere as duas e garante que o conjunto de teste é **exatamente o mesmo** nas duas organizações (mesmas 8 imagens), para que a comparação seja justa. Se a estrutura do YOLO não existir, rode antes a seção 2.1 do notebook da Entrega 1.

In [ ]:
# 2.0 Conferir as duas organizações do dataset (falha cedo se algo estiver errado)
ORIGEM = os.path.join(BASE_PATH, 'Dataset_dividido')
CLASSES = ['tomate', 'pimentao']                       # índice 0 e 1 — mesma ordem do data.yaml da Entrega 1
PASTA_CLASSE = {'tomate': 'Tomate', 'pimentao': 'Pimentao'}
SPLITS = {'treino': 'Treino', 'validacao': 'Validacao', 'teste': 'Teste'}
ESPERADO = {'treino': 32, 'validacao': 4, 'teste': 4}  # por classe
EXTENSOES = ('.jpg', '.jpeg', '.png')


def nome_normalizado(arquivo):
    # 'Tomate 16.jpg' → 'tomate_16' (mesma regra do script que gerou a estrutura do YOLO)
    return '_'.join(os.path.splitext(arquivo)[0].lower().split())


# Organização por classe → lista de (caminho, índice da classe, nome normalizado) por split
amostras = {split: [] for split in SPLITS}
problemas = []
for split, pasta_split in SPLITS.items():
    for indice, classe in enumerate(CLASSES):
        pasta = os.path.join(ORIGEM, PASTA_CLASSE[classe], pasta_split)
        arquivos = sorted(f for f in os.listdir(pasta) if f.lower().endswith(EXTENSOES))
        if len(arquivos) != ESPERADO[split]:
            problemas.append(f'{PASTA_CLASSE[classe]}/{pasta_split}: {len(arquivos)} imagens (esperado {ESPERADO[split]})')
        amostras[split] += [(os.path.join(pasta, f), indice, nome_normalizado(f)) for f in arquivos]
    print(f'Por classe  | {split:<9}: {len(amostras[split]):>2} imagens')

# Organização do YOLO → o teste precisa ter as mesmas 8 imagens, cada uma com seu rótulo
PASTA_TESTE_YOLO = os.path.join(ORIGEM, 'teste', 'images')
PASTA_ROTULOS_TESTE = os.path.join(ORIGEM, 'teste', 'labels')
if not os.path.isdir(PASTA_TESTE_YOLO) or not os.path.isdir(PASTA_ROTULOS_TESTE):
    problemas.append('Dataset_dividido/teste/images ou labels não existe — rode a seção 2.1 do notebook da Entrega 1')
else:
    teste_yolo = sorted(f for f in os.listdir(PASTA_TESTE_YOLO) if f.lower().endswith(EXTENSOES))
    nomes_yolo = {os.path.splitext(f)[0] for f in teste_yolo}
    nomes_classe = {nome for _, _, nome in amostras['teste']}
    print(f'Formato YOLO| teste    : {len(teste_yolo):>2} imagens')
    if nomes_yolo != nomes_classe:
        problemas.append(f'teste difere entre as organizações: {sorted(nomes_yolo ^ nomes_classe)}')
    sem_rotulo = [n for n in nomes_yolo if not os.path.exists(os.path.join(PASTA_ROTULOS_TESTE, n + '.txt'))]
    if sem_rotulo:
        problemas.append(f'imagens de teste sem rótulo: {sorted(sem_rotulo)}')

if problemas:
    raise RuntimeError('Problemas no dataset:\n- ' + '\n- '.join(problemas))

# Conjunto de teste comum às três abordagens (ordem fixa: 4 tomates, depois 4 pimentões)
TESTE = amostras['teste']
CAMINHO_YOLO = {os.path.splitext(f)[0]: os.path.join(PASTA_TESTE_YOLO, f) for f in teste_yolo}
CAMINHO_YOLO = {nome: CAMINHO_YOLO[nome] for _, _, nome in TESTE}   # mesma ordem do TESTE
print('\n✅ Dataset conferido. Imagens de teste:', ', '.join(nome for _, _, nome in TESTE))

In [ ]:
# 2.1 data.yaml para avaliar a YOLO customizada no conjunto de TESTE
# (mesmo conteúdo do data.yaml da Entrega 1, gravado na pasta desta entrega para não mexer no original)
DATA_YAML = f'{PASTA_E2}/data.yaml'
with open(DATA_YAML, 'w') as f:
    f.write(f'''path: {ORIGEM}
train: treino/images
val: validacao/images
test: teste/images

nc: 2
names: {CLASSES}
''')
print(open(DATA_YAML).read())

## 3. Protocolo de avaliação (igual para as três abordagens)

Para comparar modelos que fazem tarefas diferentes (detecção × classificação), usamos uma régua comum e deixamos as métricas específicas de cada um como complemento:

| Critério | Como é medido |
|:--|:--|
| **Precisão** (régua comum) | **Acurácia por imagem** nas 8 imagens de teste: a resposta do modelo para a imagem inteira está certa? Na YOLO, a resposta é a classe da detecção de maior confiança com confiança ≥ 0,5 (mesmo limiar da Entrega 1); sem detecção ≥ 0,5 conta como erro. Na CNN, a classe com probabilidade > 0,5. |
| Precisão (complemento) | YOLO customizada: mAP50 e mAP50-95 no conjunto de **teste** (a Entrega 1 só tinha mAP de validação). YOLO tradicional e YOLO customizada: **localização** — a melhor caixa prevista se sobrepõe à caixa rotulada (IoU ≥ 0,5), independentemente do nome da classe? CNN: matriz de confusão. |
| **Tempo de treino/customização** | YOLO customizada: tempo total gravado no `results.csv` da execução oficial da Entrega 1 (60 épocas, T4). YOLO tradicional: zero (só baixa os pesos). CNN: cronômetro em volta do `fit`. Os três incluem a validação a cada época. |
| **Tempo de inferência** | Mediana, em ms por imagem, de 5 rodadas sobre as 8 imagens de teste, **uma imagem por vez** (como numa câmera), após uma rodada de aquecimento. Medido de duas formas: **de ponta a ponta** — do arquivo no disco até a resposta final (leitura + pré-processamento + rede + pós-processamento), que é o que o usuário percebe — e **só a rede**, que isola o custo do modelo (nas fotos grandes do celular, só decodificar o JPEG pode levar mais tempo que a própria rede). |
| **Facilidade de uso/integração** | Análise qualitativa (seção 7.1): rotulação necessária, código, formato do modelo e integração com a câmera do "Ir Além". |

> **Limitação estatística:** com 8 imagens de teste, cada erro muda a acurácia em 12,5 pontos percentuais. Os números servem para comparar as abordagens neste dataset, não como estimativa precisa de desempenho em produção.

In [ ]:
# 3.0 Funções de apoio usadas pelas três abordagens
CONF_RESPOSTA = 0.5   # limiar para a YOLO "responder" a classe da imagem (mesmo da Entrega 1)
CONF_LISTAGEM = 0.25  # limiar padrão do Ultralytics, usado só para listar o que a YOLO enxerga


def medir_inferencia_ms(funcao, caminhos, rodadas=5):
    # Mediana do tempo (ms) de funcao(caminho), uma imagem por vez, após 1 rodada de aquecimento
    for c in caminhos:
        funcao(c)
    tempos = []
    for _ in range(rodadas):
        for c in caminhos:
            inicio = time.perf_counter()
            funcao(c)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            tempos.append((time.perf_counter() - inicio) * 1000)
    return float(np.median(tempos))


def medir_rede_yolo_ms(modelo, caminhos, rodadas=5):
    # Só a rede neural: mediana do tempo de "inference" que o próprio Ultralytics mede em cada predição
    # (sem leitura do arquivo, redimensionamento e NMS)
    modelo.predict(caminhos[0], conf=CONF_RESPOSTA, verbose=False)   # aquecimento
    tempos = [modelo.predict(c, conf=CONF_RESPOSTA, verbose=False)[0].speed['inference']
              for _ in range(rodadas) for c in caminhos]
    return float(np.median(tempos))


def caixa_rotulada(nome):
    # Caixa do rótulo de teste (formato YOLO: classe xc yc w h, normalizados) → (x1, y1, x2, y2) normalizados
    with open(os.path.join(PASTA_ROTULOS_TESTE, nome + '.txt')) as f:
        _, xc, yc, w, h = map(float, f.readline().split())
    return (xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2)


def iou(a, b):
    # Interseção sobre união entre duas caixas (x1, y1, x2, y2)
    largura = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    altura = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    intersecao = largura * altura
    uniao = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - intersecao
    return intersecao / uniao if uniao > 0 else 0.0


def avaliar_yolo(modelo, classe_esperada_por_nome):
    # Roda a YOLO nas 8 imagens de teste e monta uma linha por imagem
    linhas = []
    for _, indice, nome in TESTE:
        r = modelo.predict(CAMINHO_YOLO[nome], conf=CONF_LISTAGEM, verbose=False)[0]
        nomes = [r.names[int(c)] for c in r.boxes.cls]
        confs = [float(c) for c in r.boxes.conf]
        caixas = [tuple(map(float, b)) for b in r.boxes.xyxyn]
        gt = caixa_rotulada(nome)
        ious = [iou(c, gt) for c in caixas]
        # resposta = detecção de maior confiança, se passar do limiar
        resposta = nomes[int(np.argmax(confs))] if confs and max(confs) >= CONF_RESPOSTA else '(nenhuma ≥ 0,5)'
        linhas.append({
            'imagem': nome,
            'classe real': CLASSES[indice],
            'detecções (conf ≥ 0,25)': ', '.join(f'{n} {c:.2f}' for n, c in zip(nomes, confs)) or '—',
            'resposta': resposta,
            'acertou?': resposta == classe_esperada_por_nome(CLASSES[indice]),
            'melhor IoU': round(max(ious), 2) if ious else 0.0,
            'localizou? (IoU ≥ 0,5)': bool(ious) and max(ious) >= 0.5,
        })
    return pd.DataFrame(linhas)

## 4. Abordagem A — YOLO customizada (modelo da Entrega 1)

O modelo **não é treinado de novo**: carregamos o `best.pt` da execução oficial da Entrega 1 (60 épocas, melhor época = 48). O tempo de treino vem do `results.csv` dessa mesma execução — a coluna `time` é o tempo acumulado, em segundos, ao fim de cada época.

In [ ]:
# 4.0 Carregar o modelo da Entrega 1 e o tempo de treino registrado
if not os.path.exists(PESOS_E1):
    raise FileNotFoundError(f'best.pt da Entrega 1 não encontrado em {PESOS_E1} — confira o atalho do Drive')

modelo_custom = YOLO(PESOS_E1)
print('Classes do modelo:', modelo_custom.names)

historico_e1 = pd.read_csv(f'{PASTA_TREINO_E1}/results.csv')
historico_e1.columns = historico_e1.columns.str.strip()
TEMPO_TREINO_CUSTOM_S = float(historico_e1['time'].iloc[-1])
melhor_epoca = int(historico_e1.loc[historico_e1['metrics/mAP50-95(B)'].idxmax(), 'epoch'])
print(f'Tempo de treino registrado (60 épocas, T4): {TEMPO_TREINO_CUSTOM_S:.1f} s | melhor época: {melhor_epoca}')

In [ ]:
# 4.1 Métricas de detecção no conjunto de TESTE (inédito para o modelo: não usado no treino nem na validação)
metricas_teste = modelo_custom.val(data=DATA_YAML, split='test', project=PASTA_E2,
                                   name='val_teste_yolo_customizada', exist_ok=True, verbose=False)
MAP50_TESTE = float(metricas_teste.box.map50)
MAP5095_TESTE = float(metricas_teste.box.map)
print(f'Teste — precisão {metricas_teste.box.mp:.3f} | recall {metricas_teste.box.mr:.3f} | '
      f'mAP50 {MAP50_TESTE:.3f} | mAP50-95 {MAP5095_TESTE:.3f}')

In [ ]:
# 4.2 Resposta por imagem, localização e tempo de inferência
tabela_custom = avaliar_yolo(modelo_custom, classe_esperada_por_nome=lambda classe: classe)
display(tabela_custom)

ACURACIA_CUSTOM = tabela_custom['acertou?'].mean()
LOCALIZACAO_CUSTOM = tabela_custom['localizou? (IoU ≥ 0,5)'].mean()
INFERENCIA_CUSTOM_MS = medir_inferencia_ms(
    lambda c: modelo_custom.predict(c, conf=CONF_RESPOSTA, verbose=False), list(CAMINHO_YOLO.values()))

print(f'Acurácia por imagem: {tabela_custom["acertou?"].sum()}/8 = {ACURACIA_CUSTOM:.1%}')
print(f'Localização (IoU ≥ 0,5): {tabela_custom["localizou? (IoU ≥ 0,5)"].sum()}/8')
REDE_CUSTOM_MS = medir_rede_yolo_ms(modelo_custom, list(CAMINHO_YOLO.values()))
print(f'Inferência de ponta a ponta: {INFERENCIA_CUSTOM_MS:.1f} ms/imagem | só a rede: {REDE_CUSTOM_MS:.1f} ms (medianas)')

## 5. Abordagem B — YOLO tradicional (YOLOv8n pré-treinado no COCO, sem ajuste)

É o mesmo detector que serviu de ponto de partida para a Entrega 1, **antes** de qualquer treino com as nossas imagens. Ele foi treinado pela Ultralytics no dataset COCO, que tem 80 classes do cotidiano (pessoa, carro, cachorro, xícara…). Entre os alimentos estão `banana`, `apple`, `orange`, `broccoli`, `carrot` — mas **não há tomate nem pimentão**.

Consequência esperada: o modelo **não tem como responder "tomate" ou "pimentão"**, então a acurácia de classificação neste problema é zero por construção. A pergunta interessante é outra: **ele ao menos encontra o objeto** (caixa no lugar certo) e **com que nome** o confunde? Isso mostra o que a customização da Entrega 1 acrescentou.

**Tempo de treino:** zero — os pesos são baixados prontos (cerca de 6 MB).

In [ ]:
# 5.0 Carregar o YOLOv8n padrão (pesos genéricos do COCO) e conferir as classes que ele conhece
modelo_tradicional = YOLO('yolov8n.pt')
classes_coco = list(modelo_tradicional.names.values())
print(f'{len(classes_coco)} classes do COCO.')
print('Tem "tomato"?', 'tomato' in classes_coco, '| Tem "pepper"?', any('pepper' in c for c in classes_coco))
print('Alimentos que ele conhece:', [c for c in classes_coco if c in
      ('banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake')])
TEMPO_TREINO_TRADICIONAL_S = 0.0

In [ ]:
# 5.1 O que a YOLO tradicional enxerga nas 8 imagens de teste
# Nenhuma classe do COCO é "tomate" ou "pimentão", então nenhuma resposta pode ser considerada correta.
tabela_tradicional = avaliar_yolo(modelo_tradicional, classe_esperada_por_nome=lambda classe: None)
display(tabela_tradicional)

ACURACIA_TRADICIONAL = tabela_tradicional['acertou?'].mean()
LOCALIZACAO_TRADICIONAL = tabela_tradicional['localizou? (IoU ≥ 0,5)'].mean()
INFERENCIA_TRADICIONAL_MS = medir_inferencia_ms(
    lambda c: modelo_tradicional.predict(c, conf=CONF_RESPOSTA, verbose=False), list(CAMINHO_YOLO.values()))

print(f'Acurácia por imagem: {tabela_tradicional["acertou?"].sum()}/8 = {ACURACIA_TRADICIONAL:.1%}')
print(f'Localizou o objeto (qualquer nome, IoU ≥ 0,5): {tabela_tradicional["localizou? (IoU ≥ 0,5)"].sum()}/8')
REDE_TRADICIONAL_MS = medir_rede_yolo_ms(modelo_tradicional, list(CAMINHO_YOLO.values()))
print(f'Inferência de ponta a ponta: {INFERENCIA_TRADICIONAL_MS:.1f} ms/imagem | só a rede: {REDE_TRADICIONAL_MS:.1f} ms (medianas)')

In [ ]:
# 5.2 Prints: as 8 imagens de teste lado a lado — YOLO customizada (em cima) × YOLO tradicional (embaixo)
# As imagens anotadas também são gravadas no Drive, em Resultados/entrega2_<EXECUTOR>/predict_*/
fig, eixos = plt.subplots(2, 8, figsize=(24, 7))
for linha, (modelo, rotulo, pasta) in enumerate([(modelo_custom, 'customizada', 'predict_yolo_customizada'),
                                                 (modelo_tradicional, 'tradicional', 'predict_yolo_tradicional')]):
    resultados = modelo.predict(list(CAMINHO_YOLO.values()), conf=CONF_LISTAGEM, save=True,
                                project=PASTA_E2, name=pasta, exist_ok=True, verbose=False)
    for coluna, ((_, _, nome), r) in enumerate(zip(TESTE, resultados)):
        eixos[linha, coluna].imshow(r.plot()[:, :, ::-1])   # r.plot() devolve BGR
        eixos[linha, coluna].set_title(f'{nome}\nYOLO {rotulo}', fontsize=9)
        eixos[linha, coluna].axis('off')
plt.tight_layout()
plt.show()

## 6. Abordagem C — CNN treinada do zero

Aqui a tarefa muda: a CNN **classifica a imagem inteira** (tomate ou pimentão), sem desenhar caixas. Por isso ela não precisa das rotulações do Make Sense — a classe vem da pasta onde a foto está.

### Escolhas de arquitetura e por quê

| Escolha | Motivo |
|:--|:--|
| Entrada de **128 × 128** pixels | Suficiente para cor e formato (o que diferencia tomate de pimentão) e deixa o treino rápido. Imagens maiores aumentariam muito o número de parâmetros para só 64 fotos de treino. |
| **Aumento de dados** (espelhamento, rotação, zoom) dentro do modelo | Com 64 imagens de treino, o risco principal é decorar as fotos. Cada época vê versões ligeiramente diferentes de cada foto. As camadas de aumento só agem no treino; na predição são desligadas automaticamente. |
| **3 blocos convolução + max-pooling** (16 → 32 → 64 filtros) | Arquitetura pequena de propósito: os primeiros filtros captam bordas e cores, os seguintes, formas. Poucas camadas = poucos parâmetros = menos sobreajuste com dataset pequeno. |
| **Global Average Pooling** em vez de `Flatten` | Resume cada mapa de características em um número. Reduz drasticamente os parâmetros da parte densa (com `Flatten`, só a primeira camada densa teria centenas de milhares de pesos). |
| **Dropout 0,3** | Desliga 30% das conexões a cada passo de treino — outra defesa contra decorar o conjunto de treino. |
| Saída **1 neurônio sigmoide** + `binary_crossentropy` | Problema binário: a saída é a probabilidade de ser pimentão (classe 1); abaixo de 0,5 é tomate. |
| **Adam** (taxa 0,001) e **parada antecipada** | Adam converge bem sem ajuste fino de taxa. A parada antecipada observa a **acurácia** de validação e, se não melhorar por 30 épocas, para e restaura os pesos da época de maior acurácia — por que não a perda de validação, ver a nota antes da 6.2. |
| Semente fixa (`set_random_seed(0)`) | Mesma inicialização e mesma ordem de embaralhamento a cada execução. Na GPU ainda pode haver pequenas variações, por causa de operações não determinísticas. |

In [ ]:
# 6.0 Carregar as imagens em memória, já no tamanho da rede
TAMANHO = 128


def carregar_imagem(caminho):
    # exif_transpose aplica a rotação gravada pelo celular — sem isso, algumas fotos entram deitadas
    imagem = ImageOps.exif_transpose(Image.open(caminho)).convert('RGB')
    return np.asarray(imagem.resize((TAMANHO, TAMANHO)), dtype=np.float32)


def montar_arrays(lista):
    return np.stack([carregar_imagem(c) for c, _, _ in lista]), np.array([i for _, i, _ in lista], dtype=np.float32)


X_treino, y_treino = montar_arrays(amostras['treino'])
X_val, y_val = montar_arrays(amostras['validacao'])
X_teste, y_teste = montar_arrays(TESTE)
print('treino', X_treino.shape, '| validação', X_val.shape, '| teste', X_teste.shape)
print('pimentões no treino:', int(y_treino.sum()), 'de', len(y_treino))

In [ ]:
# 6.1 Arquitetura da CNN
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(0)

cnn = keras.Sequential([
    keras.Input(shape=(TAMANHO, TAMANHO, 3)),
    # Aumento de dados — ativo só durante o treino
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    # Pixels de 0–255 para 0–1
    layers.Rescaling(1. / 255),
    # Bloco 1: bordas e cores
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    # Bloco 2: texturas e partes do contorno
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    # Bloco 3: formas mais completas
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    # Classificador
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),   # probabilidade de ser pimentão
], name='cnn_do_zero')

cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()

### Por que a parada antecipada observa a acurácia de validação, e não a perda

Com uma validação de **só 8 imagens**, a perda de validação (`val_loss`) não é um critério de parada confiável: cada erro dado com confiança alta pesa desproporcionalmente na média, e uma ou duas imagens passam a decidir a métrica sozinhas.

Isso aconteceu na primeira execução deste notebook, que usava `val_loss` com paciência de 15 épocas:

- a menor `val_loss` foi a da **época 1** (0,694 — praticamente ln 2 = 0,693, a perda de uma rede que responde 0,5 para tudo). A partir daí ela só subiu, e o treino parou na época 16;
- os pesos restaurados eram, portanto, os de uma rede quase sem treino: P(pimentão) entre 0,50 e 0,51 em **todas** as imagens — inclusive nas 64 de treino — e todas as 8 de teste classificadas como pimentão (4/8, 50%).

Num treino de diagnóstico da mesma rede, com a mesma semente e 100 épocas sem parada, as probabilidades se separaram (tomates perto de 0, pimentões perto de 1) e a acurácia de validação subiu de 0,25 para 0,625 — enquanto a `val_loss` subia de 0,69 para 1,06, puxada por poucas imagens erradas com convicção. Ou seja: a rede aprende, mas devagar (só fica útil por volta da época 25), e a `val_loss` apontava para a época errada.

Por isso usamos a **acurácia de validação** como critério (`mode='max'`), com paciência de **30 épocas** para cobrir a faixa em que o aprendizado acontece, mantendo a restauração dos melhores pesos. A acurácia também é ruidosa com essa amostra — só anda em degraus de 12,5 p.p. (1 imagem) —, mas ao menos acompanha a evolução real do modelo. **É uma limitação conhecida do tamanho do dataset, não um bug.**

In [ ]:
# 6.2 Treino (o cronômetro mede só o fit, que inclui a validação a cada época — como no YOLO)
EPOCAS_MAX = 100
# Critério: acurácia de validação (ver a nota acima) — a val_loss com só 8 imagens não serve para escolher a época
parada = keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=30, restore_best_weights=True)

inicio = time.perf_counter()
historico_cnn = cnn.fit(X_treino, y_treino, validation_data=(X_val, y_val),
                        epochs=EPOCAS_MAX, batch_size=8, callbacks=[parada], verbose=2)
TEMPO_TREINO_CNN_S = time.perf_counter() - inicio

epocas_rodadas = len(historico_cnn.history['loss'])
# np.argmax devolve a PRIMEIRA época com a maior val_accuracy — a mesma que o Keras restaura
# (ele só troca de época quando a métrica melhora estritamente)
melhor_epoca_cnn = int(np.argmax(historico_cnn.history['val_accuracy'])) + 1
print(f'\nTreino: {TEMPO_TREINO_CNN_S:.1f} s | épocas rodadas: {epocas_rodadas} | melhor época (maior val_accuracy): {melhor_epoca_cnn}')

# Guardar o modelo e o histórico no Drive (para recarregar sem treinar de novo: keras.models.load_model(...))
cnn.save(f'{PASTA_E2}/cnn_do_zero.keras')
pd.DataFrame(historico_cnn.history).to_csv(f'{PASTA_E2}/cnn_historico.csv', index_label='epoca')

In [ ]:
# 6.3 Curvas de aprendizado
h = historico_cnn.history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(h['loss'], label='treino')
ax1.plot(h['val_loss'], label='validação')
ax1.axvline(melhor_epoca_cnn - 1, color='gray', linestyle='--', label='melhor época')
ax1.set_title('Perda (binary crossentropy)')
ax1.set_xlabel('época')
ax1.legend()
ax2.plot(h['accuracy'], label='treino')
ax2.plot(h['val_accuracy'], label='validação')
ax2.axvline(melhor_epoca_cnn - 1, color='gray', linestyle='--', label='melhor época')
ax2.set_title('Acurácia')
ax2.set_xlabel('época')
ax2.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Avaliação no conjunto de TESTE (as mesmas 8 imagens usadas nas duas YOLO)
prob_pimentao = cnn.predict(X_teste, verbose=0).ravel()
previsto = (prob_pimentao > 0.5).astype(int)

tabela_cnn = pd.DataFrame({
    'imagem': [nome for _, _, nome in TESTE],
    'classe real': [CLASSES[int(i)] for i in y_teste],
    'P(pimentão)': prob_pimentao.round(3),
    'resposta': [CLASSES[i] for i in previsto],
    'acertou?': previsto == y_teste.astype(int),
})
display(tabela_cnn)

ACURACIA_CNN = tabela_cnn['acertou?'].mean()
print(f'Acurácia por imagem: {tabela_cnn["acertou?"].sum()}/8 = {ACURACIA_CNN:.1%}\n')
print('Matriz de confusão (linhas = real, colunas = previsto; ordem tomate, pimentão):')
print(confusion_matrix(y_teste.astype(int), previsto, labels=[0, 1]))
print()
print(classification_report(y_teste.astype(int), previsto, labels=[0, 1], target_names=CLASSES, zero_division=0))

In [ ]:
# 6.5 Tempo de inferência da CNN — mesmo protocolo das YOLO (do arquivo no disco até a resposta)
def classificar_cnn(caminho):
    x = carregar_imagem(caminho)[None, ...]
    return float(cnn(x, training=False)[0, 0])   # chamada direta: menos sobrecarga que predict() para 1 imagem


INFERENCIA_CNN_MS = medir_inferencia_ms(classificar_cnn, [c for c, _, _ in TESTE])
# Só a rede: imagem já carregada e redimensionada em memória (X_teste)
REDE_CNN_MS = medir_inferencia_ms(lambda i: float(cnn(X_teste[i:i + 1], training=False)[0, 0]), list(range(len(X_teste))))
print(f'Inferência de ponta a ponta: {INFERENCIA_CNN_MS:.1f} ms/imagem | só a rede: {REDE_CNN_MS:.1f} ms (medianas)')

# Prints: as 8 imagens de teste com a resposta da CNN
fig, eixos = plt.subplots(1, 8, figsize=(24, 4))
for eixo, (caminho, indice, nome), p in zip(eixos, TESTE, prob_pimentao):
    resposta = CLASSES[int(p > 0.5)]
    eixo.imshow(ImageOps.exif_transpose(Image.open(caminho)).convert('RGB'))
    eixo.set_title(f'{nome}\nCNN: {resposta} (P(pimentão)={p:.2f})',
                   color='green' if resposta == CLASSES[indice] else 'red', fontsize=9)
    eixo.axis('off')
plt.tight_layout()
plt.savefig(f'{PASTA_E2}/cnn_teste.jpg', dpi=80)
plt.show()

## 7. Comparação das três abordagens

A tabela abaixo é montada a partir das variáveis medidas nas seções 4, 5 e 6 (nenhum número digitado à mão) e é gravada em `Resultados/entrega2_<EXECUTOR>/comparacao.csv`.

In [ ]:
# 7.0 Tabela comparativa
dispositivo = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
comparacao = pd.DataFrame({
    'A — YOLO customizada (E1)': {
        'Tarefa': 'detecção (caixa + classe)',
        'Acurácia por imagem (teste, 8 imgs)': f'{ACURACIA_CUSTOM:.1%}',
        'Localizou o objeto (IoU ≥ 0,5)': f'{LOCALIZACAO_CUSTOM:.1%}',
        'mAP50 / mAP50-95 (teste)': f'{MAP50_TESTE:.3f} / {MAP5095_TESTE:.3f}',
        'Tempo de treino/customização': f'{TEMPO_TREINO_CUSTOM_S:.0f} s (60 épocas, registrado na E1)',
        'Inferência (ms/imagem, ponta a ponta)': f'{INFERENCIA_CUSTOM_MS:.1f}',
        'Inferência (ms, só a rede)': f'{REDE_CUSTOM_MS:.1f}',
        'Rotulação necessária': 'caixas desenhadas em 64 imagens',
    },
    'B — YOLO tradicional (COCO)': {
        'Tarefa': 'detecção (80 classes do COCO)',
        'Acurácia por imagem (teste, 8 imgs)': f'{ACURACIA_TRADICIONAL:.1%} (não conhece as classes)',
        'Localizou o objeto (IoU ≥ 0,5)': f'{LOCALIZACAO_TRADICIONAL:.1%}',
        'mAP50 / mAP50-95 (teste)': 'não se aplica (classes diferentes)',
        'Tempo de treino/customização': f'{TEMPO_TREINO_TRADICIONAL_S:.0f} s (pesos prontos)',
        'Inferência (ms/imagem, ponta a ponta)': f'{INFERENCIA_TRADICIONAL_MS:.1f}',
        'Inferência (ms, só a rede)': f'{REDE_TRADICIONAL_MS:.1f}',
        'Rotulação necessária': 'nenhuma',
    },
    'C — CNN do zero': {
        'Tarefa': 'classificação (imagem inteira)',
        'Acurácia por imagem (teste, 8 imgs)': f'{ACURACIA_CNN:.1%}',
        'Localizou o objeto (IoU ≥ 0,5)': 'não se aplica (não gera caixa)',
        'mAP50 / mAP50-95 (teste)': 'não se aplica',
        'Tempo de treino/customização': f'{TEMPO_TREINO_CNN_S:.0f} s ({epocas_rodadas} épocas, parada antecipada)',
        'Inferência (ms/imagem, ponta a ponta)': f'{INFERENCIA_CNN_MS:.1f}',
        'Inferência (ms, só a rede)': f'{REDE_CNN_MS:.1f}',
        'Rotulação necessária': 'só separar as fotos em pastas por classe',
    },
})
comparacao.to_csv(f'{PASTA_E2}/comparacao.csv')
print('Dispositivo das medições de tempo:', dispositivo)
display(comparacao)

In [ ]:
# 7.1 Resposta das três abordagens para cada imagem de teste
por_imagem = pd.DataFrame({
    'imagem': [nome for _, _, nome in TESTE],
    'classe real': [CLASSES[i] for _, i, _ in TESTE],
    'A — YOLO customizada': tabela_custom['resposta'].values,
    'B — YOLO tradicional': tabela_tradicional['resposta'].values,
    'C — CNN do zero': tabela_cnn['resposta'].values,
})
por_imagem.to_csv(f'{PASTA_E2}/respostas_por_imagem.csv', index=False)
display(por_imagem)

### 7.1 Facilidade de uso e integração

| Aspecto | A — YOLO customizada | B — YOLO tradicional | C — CNN do zero |
|:--|:--|:--|:--|
| Preparar os dados | Mais trabalhoso: desenhar uma caixa em cada imagem de treino (Make Sense), exportar, parear imagem/rótulo e montar a estrutura `images/labels` + `data.yaml`. Na nossa execução, ainda foi preciso corrigir o índice de classe dos rótulos exportados. | Nenhum: funciona sem dataset. | Simples: basta separar as fotos em uma pasta por classe. |
| Código para treinar | Poucas linhas (`YOLO('yolov8n.pt').train(data=..., epochs=...)`); a biblioteca cuida de aumento de dados, métricas e gráficos. | Nenhum treino — uma linha para carregar e uma para prever. | Mais código e mais decisões: arquitetura, pré-processamento, aumento de dados, função de perda, parada antecipada, métricas — tudo escrito à mão. |
| Ajustar ao problema | Bom equilíbrio: aproveita o que o modelo já sabe (COCO) e aprende as nossas classes com poucas imagens. | Impossível sem treinar: só reconhece as 80 classes do COCO. | Total controle, mas aprende tudo do zero — com poucas imagens, depende muito do aumento de dados. |
| Saída | Classe + posição (caixa) + confiança — permite contar objetos e saber onde estão. | Idem, mas com as classes do COCO. | Só a classe da imagem inteira; não diz onde o objeto está nem lida bem com tomate e pimentão na mesma foto. |
| Integração (câmera do "Ir Além 1") | Direta: o mesmo `best.pt` roda em webcam/stream com `model.predict(source=...)`. Exporta para ONNX/TFLite. | Igual à A. | Exige escrever a captura e o pré-processamento; o `.keras` exporta para TFLite (microcontroladores/celular). |

## 8. Análise crítica e conclusões

> **Esta seção é redigida depois da execução oficial no Colab**, a partir dos números das seções 4 a 7 — nenhum valor é escrito antes de ser medido. Roteiro da análise:
>
> 1. **Precisão:** acurácia por imagem das três abordagens nas mesmas 8 imagens; o que o mAP de teste da YOLO customizada acrescenta à acurácia; onde cada uma errou (tabela 7.1).
> 2. **YOLO tradicional:** com que classes do COCO ela confundiu tomates e pimentões e se, mesmo sem saber o nome, ela localizou o objeto — o que mostra o valor da customização.
> 3. **Tempo de treino × tempo de inferência:** custo de customizar cada modelo e velocidade de resposta por imagem na T4.
> 4. **Facilidade de uso:** esforço de rotulação e de código (seção 7.1).
> 5. **Recomendação para a FarmTech:** qual abordagem usar em cada cenário (detecção em câmera, triagem simples por imagem, protótipo rápido sem dados).
> 6. **Limitações:** 8 imagens de teste (cada erro = 12,5 p.p.), uma única execução de cada treino, fotos de um mesmo contexto de captura, validação de 8 imagens como critério de parada da CNN (nota antes da 6.2).